In [391]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import DataConversionWarning
from sklearn.preprocessing import RobustScaler
from dowhy import CausalModel
from IPython.display import display


warnings.filterwarnings("ignore", category=DataConversionWarning)

In [392]:
df = pd.read_csv("alerts_with_category_with_apk_size_updated.csv")
print("Original rows:", len(df))

df = df[~df["code_location"].str.contains("obfuscated", case=False, na=False)].copy()
print("After removing obfuscated:", len(df))

df["verdict"] = df["verdict"].astype(int)

download_order = ['<100','100-500','500-1k','1k-5k','5k-10k','10k-50k',
                  '50k-100k','100k-500k','500k-1M','1M-5M','>5M']
ord_map = {c:i for i,c in enumerate(download_order)}
df["app_popularity_encoded"] = df["app_popularity"].map(ord_map)
df = df.dropna(subset=["app_popularity_encoded"]).copy()
df["app_popularity_encoded"] = df["app_popularity_encoded"].astype(int)

scaler = RobustScaler()
df["apk_size_scaled"] = scaler.fit_transform(df[["apk_size"]])


Original rows: 6140
After removing obfuscated: 5612


In [393]:
small_libs = ["SocialMedia", "Analytics", "Cloud"]
df["lib_grouped"] = df["code_location"].replace(small_libs, "Small_Libraries")
df.loc[df["lib_grouped"] == "developer_written", "lib_grouped"] = "App_Source_Code"

print("\nGrouped library counts:")
print(df["lib_grouped"].value_counts())

baseline_name = "App_Source_Code"
libs = [baseline_name] + [l for l in df["lib_grouped"].unique() if l != baseline_name]
print("\nLibrary levels:")
for l in libs:
    print("-", l)


Grouped library counts:
lib_grouped
Utilities          3039
others             1341
App_Source_Code     511
Android             494
Small_Libraries     227
Name: count, dtype: int64

Library levels:
- App_Source_Code
- others
- Android
- Utilities
- Small_Libraries


In [394]:
refutation_methods = [
    "random_common_cause",
    "placebo_treatment_refuter",
    "data_subset_refuter",
]

In [395]:
def make_pairwise_balanced(df_in,lib_name,baseline="App_Source_Code",random_state=100):
    df_sub = df_in[df_in["lib_grouped"].isin([baseline, lib_name])].copy()

    base = df_sub[df_sub["lib_grouped"] == baseline]
    lib  = df_sub[df_sub["lib_grouped"] == lib_name]

    n_base = len(base)
    n_lib  = len(lib)

    base_bal = base.copy()

    if n_lib >= n_base:
        lib_bal = lib.sample(n=n_base, replace=False, random_state=random_state)
    else:
        lib_bal = lib.sample(n=n_base, replace=True, random_state=random_state)

    df_bal = pd.concat([base_bal, lib_bal], axis=0)
    df_bal = df_bal.sample(frac=1, random_state=random_state).reset_index(drop=True)
   
    return df_bal



In [396]:
def apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate):
    ref_rows = []
    for method in refutation_methods:
        if method == "placebo_treatment_refuter":
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method, placebo_type="permute", num_simulations=200)
        elif method == "data_subset_refuter":
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method, subset_fraction=0.8, num_simulations=200)
        else:
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method)

        print(f"\nRefuter: {method}\n", ref)

        ref_rows.append({
            "library": lib_name,
            "setting": label,
            "refuter": method,
            "orig_effect": ate,
            "new_effect": getattr(ref, "new_effect", None),
            "p_value": getattr(ref, "p_value", None)
        })
        
    return ref_rows

In [397]:
def run_library_contrast(df_sub, lib_name, baseline="App_Source_Code", label="imbalanced"):
    df_sub = df_sub.copy()
    df_sub["treatment_lib"] = (df_sub["lib_grouped"] == lib_name).astype(int)

    n0 = int((df_sub["treatment_lib"] == 0).sum())
    n1 = int((df_sub["treatment_lib"] == 1).sum())

    print("\n" + "-"*14 + f" {lib_name} vs {baseline} " + "-"*14)
    print("Counts:", {baseline: n0, lib_name: n1})


    causal_graph = """
    digraph {
        treatment_lib -> verdict;
        app_popularity_encoded -> treatment_lib;
        app_popularity_encoded -> verdict;
        apk_size_scaled -> treatment_lib;
       app_popularity_encoded -> apk_size_scaled;
    }
    """

    model = CausalModel(
        data=df_sub,
        treatment="treatment_lib",
        outcome="verdict",
        graph=causal_graph.replace("\n", " "),
        common_causes=["app_popularity_encoded"]
    )

    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.propensity_score_matching",
        target_units="ate",
        confidence_intervals='bootstrap',
        method_params={
            "num_simulations": 300,
            "sample_size_fraction": 1.0,
            "confidence_level": 0.95,
        },
    )

    ate = float(estimate.value)
    try:
        ci_low, ci_high = estimate.get_confidence_intervals()
        ci_low, ci_high = float(ci_low), float(ci_high)
    except Exception:
        ci_low, ci_high = None, None

    print("ATE:", ate)
    if ci_low is not None:
        print("95% CI:", (ci_low, ci_high))


    ref_rows = apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate)

    result_row = {
        "library": lib_name,
        "setting": label,
        "n_baseline": n0,
        "n_lib": n1,
        "ATE": ate,
        "CI_low": ci_low,
        "CI_high": ci_high
    }

    return result_row, ref_rows




In [398]:
main_rows = []
ref_rows_all = []

for lib in libs:
    if lib == baseline_name:
        continue

    df_bal = make_pairwise_balanced(df, lib, baseline=baseline_name, random_state=100)
    main_row_b, ref_b = run_library_contrast(df_bal, lib, baseline=baseline_name, label="balanced")
    main_rows.append(main_row_b)
    ref_rows_all.extend(ref_b)



-------------- others vs App_Source_Code --------------
Counts: {'App_Source_Code': 511, 'others': 511}
ATE: -0.19275929549902152
95% CI: (-0.5117416829745597, -0.061643835616438325)

Refuter: random_common_cause
 Refute: Add a random common cause
Estimated effect:-0.19275929549902152
New effect:-0.1927592954990216
p value:1.0


Refuter: placebo_treatment_refuter
 Refute: Use a Placebo Treatment
Estimated effect:-0.19275929549902152
New effect:-0.0046868884540117416
p value:0.925


Refuter: data_subset_refuter
 Refute: Use a subset of data
Estimated effect:-0.19275929549902152
New effect:-0.10922982885085573
p value:0.48


-------------- Android vs App_Source_Code --------------
Counts: {'App_Source_Code': 511, 'Android': 511}
ATE: 0.07827788649706457
95% CI: (-0.15166340508806264, 0.21428571428571427)

Refuter: random_common_cause
 Refute: Add a random common cause
Estimated effect:0.07827788649706457
New effect:0.07827788649706456
p value:1.0


Refuter: placebo_treatment_refuter
 Re

In [399]:
main_df = pd.DataFrame(main_rows).sort_values(["library", "setting"])
ref_df = pd.DataFrame(ref_rows_all)

print("\n\n","-"*25 + "Detailed (ATEs) " + "-"*25)
print(main_df[["library","setting","n_baseline","n_lib","ATE","CI_low","CI_high"]])

pivot = main_df.pivot(index="library", columns="setting", values="ATE")
print("\n\n","-"*25 + "Summary ATE " + "-"*25)
print(pivot)



 -------------------------Detailed (ATEs) -------------------------
           library   setting  n_baseline  n_lib       ATE    CI_low   CI_high
1          Android  balanced         511    511  0.078278 -0.151663  0.214286
3  Small_Libraries  balanced         511    511  0.349315  0.253425  0.567515
2        Utilities  balanced         511    511 -0.410959 -0.663405 -0.258317
0           others  balanced         511    511 -0.192759 -0.511742 -0.061644


 -------------------------Summary ATE -------------------------
setting          balanced
library                  
Android          0.078278
Small_Libraries  0.349315
Utilities       -0.410959
others          -0.192759
